In [ ]:
import os
from glob import glob
import random
from tqdm import tqdm
from datetime import datetime
from collections import OrderedDict, Counter
from typing import Iterable, List, Optional, Union

# font_fname = os.environ.get('PROJECT_FONT_PATH', '') #적용할 폰트
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
import seaborn as sns
from matplotlib import cm
from matplotlib.colors import Normalize
from IPython.display import display, HTML

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM

# from torchtext.vocab import Vocab

from src.utils.preprocessing import Preprocessor
from src.utils import save_dir_setup, compute_alpha
from src.models import SupervisedTextDataset, SupervisedPLModel, SimpleClassifier, get_model_predictions

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from umap import UMAP

import pytorch_lightning as pl

from captum.attr import LayerIntegratedGradients, TokenReferenceBase, visualization, IntegratedGradients



class SimpleVocab:
    def __init__(self, itos, unk_token="<unk>"):
        self.itos = list(itos)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.unk = unk_token
        if self.unk not in self.stoi:
            self.stoi[self.unk] = len(self.itos)
            self.itos.append(self.unk)
    def __len__(self): return len(self.itos)
    def to_indices(self, tokens):
        unk_id = self.stoi[self.unk]
        return [self.stoi.get(t, unk_id) for t in tokens]
    def to_tokens(self, ids):
        return [self.itos[i] if 0 <= i < len(self.itos) else self.unk for i in ids]


def build_vocab_from_iterator(iterator, min_freq=1, specials=("<pad>", "<unk>")):
    freq = Counter()
    for tokens in iterator:
        freq.update(tokens)
    itos = list(specials) + [t for t,c in freq.items() if c >= min_freq and t not in specials]
    return SimpleVocab(itos, unk_token="<unk>")



# Run IG on embeddings (not token IDs) to avoid float indices into embedding lookup
@torch.no_grad()
def predict_proba(encoded, pl_model):
    pl_model.eval()
    device = next(pl_model.parameters()).device
    return pl_model(**{k: v.to(device) for k, v in encoded.items()})


def forward_with_softmax(tokens, model):
    return torch.softmax(model(**{k:v.to(model.device) for k,v in tokens.items()}), dim = 1)


def forward_embeds(embeddings, attention_mask):
    device = next(pl_model.parameters()).device
    embeddings = embeddings.to(device)
    if attention_mask is not None:
        attention_mask = attention_mask.to(device)
    # Encoder forward with embeddings
    enc_out = pl_model.encoder(inputs_embeds=embeddings, attention_mask=attention_mask)
    if hasattr(enc_out, "pooler_output") and enc_out.pooler_output is not None:
        pooled = enc_out.pooler_output
    else:
        pooled = enc_out.last_hidden_state[:, 0, :]
    logits = pl_model.classifier(pooled)
    return logits


def add_attributions_to_visualizer(attributions, text, pred_prob, pred, label, delta, vis_data_records):
    attributions = attributions.sum(dim=2).squeeze(0)
    attributions = attributions / torch.norm(attributions)
    attributions = attributions.cpu().detach().numpy()

    # storing couple samples in an array for visualization purposes
    vis_data_records.append(visualization.VisualizationDataRecord(
                            attributions,
                            pred_prob,
                            pred,
                            label,
                            1,
                            attributions.sum(),
                            text,
                            delta))




class SimpleVocab:
    def __init__(self, itos: List[str]):
        self.itos = list(itos)
        self.stoi = {s: i for i, s in enumerate(self.itos)}
    def __len__(self): return len(self.itos)


class SimpleLabelField:
    """
    Minimal replacement for torchtext.data.LabelField.
    - build_vocab: learn unique label set (in a stable order)
    - numericalize: convert labels -> tensor with chosen dtype
    """
    def __init__(self, dtype: torch.dtype = torch.long, classes: Optional[Iterable[str]] = None):
        self.dtype = dtype
        self.vocab: Optional[SimpleVocab] = None
        # 고정 클래스 목록을 주고 싶으면 classes로 지정 (예: ["neg","pos"])
        if classes is not None:
            self.vocab = SimpleVocab(list(classes))

    def build_vocab(self, labels: Iterable[Union[str, int, float]]):
        if self.vocab is not None:
            return self  # 이미 고정 클래스가 있으면 패스
        uniq = list(OrderedDict.fromkeys(map(str, labels)))  # 순서 보존 고유값
        self.vocab = SimpleVocab(uniq)
        return self

    def numericalize(self, labels: Iterable[Union[str, int, float]]) -> torch.Tensor:
        if self.vocab is None:
            # vocab이 없으면 숫자형(0/1 등) 입력을 그대로 사용
            vals = list(labels)
            t = torch.tensor(vals, dtype=self.dtype)
            return t
        # 문자열 레이블 -> 인덱스
        idxs = []
        for y in labels:
            y_str = str(y)
            if y_str not in self.vocab.stoi:
                raise KeyError(f"Unknown label '{y_str}'. Known: {list(self.vocab.stoi.keys())}")
            idxs.append(self.vocab.stoi[y_str])

        t = torch.tensor(idxs, dtype=torch.long)
        # BCE(float)로 쓰고 싶다면 0/1 float로 변환
        if self.dtype in (torch.float, torch.float32, torch.float64, torch.bfloat16, torch.float16):
            if len(self.vocab) == 2:
                t = t.to(torch.float32)  # [0., 1.]
            else:
                # 다중분류인데 float를 원한다면 one-hot 등으로 바꿔 써야 함
                raise ValueError(
                    "Multi-class with float dtype: use torch.long for CrossEntropyLoss "
                    "or convert to one-hot manually."
                )
        else:
            t = t.to(self.dtype)
        return t

    def to_tokens(self, indices: Iterable[int]) -> List[str]:
        assert self.vocab is not None, "build_vocab first"
        out = []
        for i in indices:
            if 0 <= i < len(self.vocab):
                out.append(self.vocab.itos[i])
            else:
                out.append("<unk_label>")
        return out



negative_patterns = ['without','not? (sugges|compat)', 'unlike', 'neither', 'no', 'negative']
uncertain_patterns = ['rule out', 'Rule Out', "Rule out"]


def check_label(df, s):
    df = df.to_frame()
    df['label'] = s
    return df

# seed = 42
# batch_size = 32
# patience = 5
target = 'metas'


# save_dir_version = 'v002'
# inter_fix = '2509'
# save_dir_prefix = f'CLF-{target}-{inter_fix}'

font_fname = os.environ.get('PROJECT_FONT_PATH', '')  # point at any local CJK/serif .ttf

font_family = fm.FontProperties(fname=font_fname).get_name() #폰트 설정
plt.rcParams["font.family"] = font_family  #폰트 적용

# data_dir_path = sorted(glob(os.path.join('data','reviewed_labels',f'*_{target}.csv')))
# data_dir_path = sorted(glob(os.path.join('data','reviewed_labels',f'{target}*')))
# print(data_dir_path)

label_dict = {'negative': 0, 'uncertain': 1, 'positive': 2}
inv_label_dict = {v: k for k, v in label_dict.items()}


data_dir_path = sorted(glob(os.path.join('data','reviewed_labels','*')))
error_data_dir_path = sorted(glob(os.path.join('outputs','predictions','R2_R1_*')))


# ckpt_path_dict = dict(
#     metas = glob(os.path.join('outputs','checkpoints','CLF-meta-pubmedbert-v005','files','*epoch*.ckpt'))[0],
#     recur = glob(os.path.join('outputs','checkpoints','CLF-recur-pubmedbert-v002','files','*epoch*.ckpt'))[0]
# )

In [ ]:

model_dict = dict(
    recur = 'PubMedBERT-base-uncased-sts-combined',
    metas = 'MedEmbed-base-v0.1',
)
model_dir_path = os.path.join('..','..','..','..','model')

## raw text

raw_data_dir = glob(os.path.join('data','reports','*'))
print(raw_data_dir)
df = pd.read_csv(raw_data_dir[0], encoding = 'utf-8-sig', engine = 'c').reset_index()

text_df = df['검사결과결론내용'].dropna()

preprocessor = Preprocessor(
    df = text_df,
    negative_patterns = negative_patterns,
    uncertain_patterns = uncertain_patterns
)



In [ ]:

rev_dir_path = os.path.join('data','reviewer')
metas_df, recur_df = [pd.read_excel(i, sheet_name=None) for i in sorted(glob(os.path.join(rev_dir_path,'*.xlsx')))]

metas_labeled_df = pd.concat([metas_df[c] for c in ['negative','uncertain','positive']]).dropna()
recur_df['negative'].loc[:99, '실제'] = 'negative'
recur_labeled_df = pd.concat([recur_df[c] for c in ['negative','uncertain','positive']]).dropna()

In [ ]:


target_pred_dict = {}


for target in ['recur', 'metas']:
    
    # data_dict = preprocessor.target_filtering(target)
    # target_df = pd.concat([check_label(d, k) for k, d in data_dict.items()]).sort_index()
    # target_df.shape, target_df.drop_duplicates('검사결과결론내용').shape

    if target == 'recur':
        target_df = recur_labeled_df.copy()
    else:
        target_df = metas_labeled_df.copy()


    ckpt_dir = glob(os.path.join('outputs','checkpoints',f'*{target}*', 'files','*epoch*'))[-1]

    model_path = model_dict[target]

    tokenizer = AutoTokenizer.from_pretrained(os.path.join(model_dir_path, model_path))
    language_model = AutoModel.from_pretrained(
            os.path.join(model_dir_path, model_path),
            # dtype="bfloat16",
        )

        
    # Lightning model
    pl_model = SupervisedPLModel.load_from_checkpoint(
        ckpt_dir,
        encoder=language_model,
        hidden_dim=language_model.config.hidden_size,
        num_classes=len(label_dict),
        # lr=1e-5,
        # gamma=2.0,
        # alpha=alpha
    )

    token_reference = TokenReferenceBase(reference_token_idx=tokenizer.pad_token_id)
    lig = LayerIntegratedGradients(pl_model, pl_model.encoder.get_input_embeddings())

    # break


In [ ]:
dl = DataLoader(SupervisedTextDataset(target_df, tokenizer, 'prep_text', None), batch_size = 8, shuffle = False)

pl_model.eval()
pl_model.cuda()

logits = []
for b in dl:
    with torch.no_grad():
        logits.append(pl_model(**{k: v.cuda() for k, v in b.items()}))

probs = torch.cat(logits, dim = 0).softmax(1).max(1).values.cpu().numpy()
argmax = torch.cat(logits, dim = 0).argmax(1).cpu().numpy()

target_df['probs'] = probs
target_df['pred'] = argmax

In [ ]:
# for c in ['negative','uncertain','positive']:
for c in ['positive']:
    target_class_df = target_df.query(f"실제 == '{c}'")

    sorted_idx = target_class_df.probs.argsort()
    low_idx = sorted_idx[:3]
    high_idx = sorted_idx[-3:]

    idx = low_idx.tolist() + high_idx.tolist()
    idx = idx[::-1]
    vis_case = target_class_df.iloc[idx]
    
    vis_data_records = []
    for n, samples in vis_case.iterrows():
        text = samples['prep_text']
        label = samples['실제']

        tokens = tokenizer(text,return_tensors="pt",)

        with torch.no_grad():
            logits = predict_proba(tokens, pl_model)
            probs = logits[0].softmax(0)

            pred = forward_with_softmax(tokens, pl_model)
            pred_prob = pred[0,pred.argmax()].item()
        


        

        pl_model.zero_grad()



        reference_indices = token_reference.generate_reference(tokens['input_ids'].shape[1], device=pl_model.device).unsqueeze(0)
        attributions_ig, delta = lig.attribute(
            tokens['input_ids'].to(pl_model.device), 
            reference_indices,
            n_steps=500, 
            return_convergence_delta=True,
            target=label_dict[label] if label in label_dict else 0
        )

        # === 여기서 바로 CPU로 옮기고 그래프 끊기 ===
        attributions_ig = attributions_ig.detach().cpu()
        delta = float(delta.detach().cpu())

        tokens_for_vis = tokenizer.convert_ids_to_tokens(tokens['input_ids'][0].cpu())

        print(
            'pred: ', inv_label_dict[pred.argmax().item()],
            '(', '%.2f'%pred_prob, ')',
            ', delta: ', abs(delta)
        )

        vis_data_records.append(
            add_attributions_to_visualizer(
                attributions_ig,
                tokens_for_vis,
                pred_prob,
                inv_label_dict[pred.argmax(1)[0].item()][:3].capitalize(),
                label[:3].capitalize(),
                delta,
                vis_data_records
            )
        )

        # 필요 없다면 여기서도 정리 가능
        del attributions_ig

        torch.cuda.empty_cache()
        # gc.collect()


        # # generate reference indices for each sample
        # reference_indices = token_reference.generate_reference(tokens['input_ids'].shape[1], device=pl_model.device).unsqueeze(0)

        # # compute attributions and approximation delta using layer integrated gradients
        # attributions_ig, delta = lig.attribute(
        #     tokens['input_ids'].to(pl_model.device), 
        #     reference_indices.to(pl_model.device),
        #     n_steps=500, 
        #     return_convergence_delta=True,
        #     target = label_dict[label] if label in label_dict else 0
        #     # target = label
        # )

        # print('pred: ', inv_label_dict[pred.argmax().item()], '(', '%.2f'%pred_prob, ')', ', delta: ', abs(delta))

        # vis_data_records.append(
        #     add_attributions_to_visualizer(
        #         attributions_ig, 
        #         tokenizer.convert_ids_to_tokens(tokens['input_ids'][0]), 
        #         pred_prob, 
        #         inv_label_dict[pred.argmax(1)[0].item()][:3].capitalize(), 
        #         label[:3].capitalize(), 
        #         delta, 
        #         vis_data_records
        #     )
        # )
    print('Visualize attributions based on Integrated Gradients')
    _ = visualization.visualize_text(list(filter(None, vis_data_records)))

    # HTML 폰트를 Times(serif)로 변경
    custom_font = "font-family: 'Times New Roman', Times, serif;"
    style = f"<style>body, span, td, th, div, .attribution-text {{ {custom_font} }}</style>"

    html_data = _.data
    if "<head>" in html_data:
        html_data = html_data.replace("<head>", "<head>" + style)
    else:
        html_data = style + html_data

    with open(os.path.join('.','figure',f'{target}_{c}_attrs.html'), "w", encoding="utf-8") as f:
        f.write(_.data)  # HTML의 원본 문자열

In [ ]:
from sympy.geometry.entity import N


n = 0
samples = vis_case.reset_index(drop=True).iloc[n]
i = samples['index']
samples

text = samples['prep_text']
label = samples['실제']

tokens = tokenizer(text,return_tensors="pt",)

with torch.no_grad():
    logits = predict_proba(tokens, pl_model)
    # probs = logits[0].softmax(0)

    pred = forward_with_softmax(tokens, pl_model)
    print(logits, pred)
    # pred_prob = pred[0,pred.argmax()].item()
    # print(probs, pred_prob)

vis_data_records = []
for pred, pred_prob in enumerate(pred[0]):




    pl_model.zero_grad()
    reference_indices = token_reference.generate_reference(tokens['input_ids'].shape[1], device=pl_model.device).unsqueeze(0)
    attributions_ig, delta = lig.attribute(
        tokens['input_ids'].to(pl_model.device), 
        reference_indices,
        n_steps=500, 
        return_convergence_delta=True,
        # target=label_dict[label] if label in label_dict else 0
        target=pred
    )

    # === 여기서 바로 CPU로 옮기고 그래프 끊기 ===
    attributions_ig = attributions_ig.detach().cpu()
    delta = float(delta.detach().cpu())

    tokens_for_vis = tokenizer.convert_ids_to_tokens(tokens['input_ids'][0].cpu())

    print(
        'pred: ', inv_label_dict[pred],
        '(', '%.2f'%pred_prob, ')',
        ', delta: ', abs(delta)
    )

    vis_data_records.append(
        add_attributions_to_visualizer(
            attributions_ig,
            tokens_for_vis,
            pred_prob,
            inv_label_dict[pred][:3].capitalize(),
            label[:3].capitalize(),
            delta,
            vis_data_records
        )
    )

    # 필요 없다면 여기서도 정리 가능
    del attributions_ig

    torch.cuda.empty_cache()
print('Visualize attributions based on Integrated Gradients')
_ = visualization.visualize_text(list(filter(None, vis_data_records)))

# HTML 폰트를 Times(serif)로 변경
custom_font = "font-family: 'Times New Roman', Times, serif;"
style = f"<style>body, span, td, th, div, .attribution-text {{ {custom_font} }}</style>"

html_data = _.data
if "<head>" in html_data:
    html_data = html_data.replace("<head>", "<head>" + style)
else:
    html_data = style + html_data

with open(os.path.join('.', 'figure', f'{target}_{c}_index_{n}_attrs.html'), "w", encoding="utf-8") as f:
    f.write(html_data)

In [ ]:
label_dict

In [ ]:
pred

In [ ]:
target_df = metas_errors_sorted if target == 'metas' else recur_errors_sorted

vis_data_records = []
for n, samples in target_df.iterrows():
    text = samples['검사결과결론내용']
    label = samples['label_y']

    tokens = tokenizer(text,return_tensors="pt",)

    model.zero_grad()
    logits = predict_proba(tokens, pl_model)
    probs = logits[0].softmax(0)

    pred = forward_with_softmax(tokens, pl_model)
    pred_prob = pred[0,pred.argmax()].item()
    
    # generate reference indices for each sample
    reference_indices = token_reference.generate_reference(tokens['input_ids'].shape[1], device=pl_model.device).unsqueeze(0)

    # compute attributions and approximation delta using layer integrated gradients
    attributions_ig, delta = lig.attribute(
        tokens['input_ids'].to(pl_model.device), 
        reference_indices.to(pl_model.device),
        n_steps=500, 
        return_convergence_delta=True,
        target = label_dict[label] if label in label_dict else 0
        # target = label
    )

    print('pred: ', inv_label_dict[pred.argmax().item()], '(', '%.2f'%pred_prob, ')', ', delta: ', abs(delta))

    vis_data_records.append(
        add_attributions_to_visualizer(
            attributions_ig, 
            tokenizer.convert_ids_to_tokens(tokens['input_ids'][0]), 
            pred_prob, 
            inv_label_dict[pred.argmax(1)[0].item()][:3].capitalize(), 
            label[:3].capitalize(), 
            delta, 
            vis_data_records
        )
    )
print('Visualize attributions based on Integrated Gradients')
_ = visualization.visualize_text(list(filter(None, vis_data_records)))


with open(os.path.join('.','logs',f'{target}_attrs.html'), "w", encoding="utf-8") as f:
    f.write(_.data)  # HTML의 원본 문자열